#Configuration

In [0]:
STORAGE_ACCOUNT = "cmsmedicaredatastorage"
CONTAINER = "cms-medicare-raw"
SOURCE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/provider_utilization/"
CHECKPOINT_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/_checkpoints/bronze_provider/"
TARGET_TABLE = "cms_medicare_databricks_pipeline.bronze.provider_utilization_raw"

#Authentication
STORAGE_KEY = dbutils.secrets.get(
    scope="cms-medicare-scope",
    key="adls-storage-key"
)

#Register Key
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

print("Storage authentication configured successfully")

Storage authentication configured successfully


#Auto Loader

In [0]:

df_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "/schema")
    .load(SOURCE_PATH)
)

#Add metadata columns
For debugging and auditing

In [0]:
from pyspark.sql import functions as F

df_with_metadata = df_raw.select(
    "*",
    F.current_timestamp().alias("ingestion_timestamp"),
    F.input_file_name().alias("source_file")
)

#Write to Delta Table

In [0]:
(
    df_with_metadata
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
)

#Verify

In [0]:
count = spark.read.table(TARGET_TABLE).count()
print(f"Total rows ingested: {count:,}")

display(spark.read.table(TARGET_TABLE).limit(5))
spark.table(TARGET_TABLE).printSchema()

Total rows ingested: 9,781,673


Rndrng_NPI,Rndrng_Prvdr_Last_Org_Name,Rndrng_Prvdr_First_Name,Rndrng_Prvdr_MI,Rndrng_Prvdr_Crdntls,Rndrng_Prvdr_Ent_Cd,Rndrng_Prvdr_St1,Rndrng_Prvdr_St2,Rndrng_Prvdr_City,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_State_FIPS,Rndrng_Prvdr_Zip5,Rndrng_Prvdr_RUCA,Rndrng_Prvdr_RUCA_Desc,Rndrng_Prvdr_Cntry,Rndrng_Prvdr_Type,Rndrng_Prvdr_Mdcr_Prtcptg_Ind,HCPCS_Cd,HCPCS_Desc,HCPCS_Drug_Ind,Place_Of_Srvc,Tot_Benes,Tot_Srvcs,Tot_Bene_Day_Srvcs,Avg_Sbmtd_Chrg,Avg_Mdcr_Alowd_Amt,Avg_Mdcr_Pymt_Amt,Avg_Mdcr_Stdzd_Amt,_rescued_data,ingestion_timestamp,source_file
1821288143,Owen,Trevor,M,MD,I,3 Riverside Cir,null,Roanoke,VA,51,24016,1,"Metropolitan area core: primary flow within an urbanized area of 50,000 and greater",US,Orthopedic Surgery,Y,99214,"Established patient office or other outpatient visit with moderate level of decision making, if using time, 30 minutes or more",N,F,78,85,85,91.082352941,86.006823529,64.098352941,71.111529412,null,2026-07-04T23:06:14.91Z,abfss://cms-medicare-raw@cmsmedicaredatastorage.dfs.core.windows.net/provider_utilization/PHY_R26_P05_V10_D24_Prov_Svc.csv
1821288143,Owen,Trevor,M,MD,I,3 Riverside Cir,null,Roanoke,VA,51,24016,1,"Metropolitan area core: primary flow within an urbanized area of 50,000 and greater",US,Orthopedic Surgery,Y,99221,"Initial hospital care with straightforward or low level of medical decision making, per day, if using time, at least 40 minutes",N,F,11,11,11,237,75.612727273,61.08,65.468181818,null,2026-07-04T23:06:14.91Z,abfss://cms-medicare-raw@cmsmedicaredatastorage.dfs.core.windows.net/provider_utilization/PHY_R26_P05_V10_D24_Prov_Svc.csv
1821288143,Owen,Trevor,M,MD,I,3 Riverside Cir,null,Roanoke,VA,51,24016,1,"Metropolitan area core: primary flow within an urbanized area of 50,000 and greater",US,Orthopedic Surgery,Y,99222,"Initial hospital care with straightforward or low-level medical decision making, if using time, at least 55 minutes",N,F,56,56,56,300,124.4325,96.8475,98.62625,null,2026-07-04T23:06:14.91Z,abfss://cms-medicare-raw@cmsmedicaredatastorage.dfs.core.windows.net/provider_utilization/PHY_R26_P05_V10_D24_Prov_Svc.csv
1821288143,Owen,Trevor,M,MD,I,3 Riverside Cir,null,Roanoke,VA,51,24016,1,"Metropolitan area core: primary flow within an urbanized area of 50,000 and greater",US,Orthopedic Surgery,Y,99284,Emergency department visit with moderate level of medical decision making,N,F,29,29,29,369.10344828,114.95551724,91.594827586,93.531724138,null,2026-07-04T23:06:14.91Z,abfss://cms-medicare-raw@cmsmedicaredatastorage.dfs.core.windows.net/provider_utilization/PHY_R26_P05_V10_D24_Prov_Svc.csv
1821288143,Owen,Trevor,M,MD,I,3 Riverside Cir,null,Roanoke,VA,51,24016,1,"Metropolitan area core: primary flow within an urbanized area of 50,000 and greater",US,Orthopedic Surgery,Y,G0180,"Physician or allowed practitioner certification for medicare-covered home health services under a home health plan of care (patient not present), including contacts with home health agency and review of reports of patient status required by physicians and",N,F,24,26,26,110.30769231,51.820769231,41.290384615,41.939230769,null,2026-07-04T23:06:14.91Z,abfss://cms-medicare-raw@cmsmedicaredatastorage.dfs.core.windows.net/provider_utilization/PHY_R26_P05_V10_D24_Prov_Svc.csv


root
 |-- Rndrng_NPI: string (nullable = true)
 |-- Rndrng_Prvdr_Last_Org_Name: string (nullable = true)
 |-- Rndrng_Prvdr_First_Name: string (nullable = true)
 |-- Rndrng_Prvdr_MI: string (nullable = true)
 |-- Rndrng_Prvdr_Crdntls: string (nullable = true)
 |-- Rndrng_Prvdr_Ent_Cd: string (nullable = true)
 |-- Rndrng_Prvdr_St1: string (nullable = true)
 |-- Rndrng_Prvdr_St2: string (nullable = true)
 |-- Rndrng_Prvdr_City: string (nullable = true)
 |-- Rndrng_Prvdr_State_Abrvtn: string (nullable = true)
 |-- Rndrng_Prvdr_State_FIPS: string (nullable = true)
 |-- Rndrng_Prvdr_Zip5: string (nullable = true)
 |-- Rndrng_Prvdr_RUCA: string (nullable = true)
 |-- Rndrng_Prvdr_RUCA_Desc: string (nullable = true)
 |-- Rndrng_Prvdr_Cntry: string (nullable = true)
 |-- Rndrng_Prvdr_Type: string (nullable = true)
 |-- Rndrng_Prvdr_Mdcr_Prtcptg_Ind: string (nullable = true)
 |-- HCPCS_Cd: string (nullable = true)
 |-- HCPCS_Desc: string (nullable = true)
 |-- HCPCS_Drug_Ind: string (nullable =